In [1]:
from pyscf import gto, fci, mcscf
import numpy as np
from gpu4pyscf import scf


/home/mert/.local/lib/python3.10/site-packages/gpu4pyscf/lib/cutensor.py:154: UserWarning: using cupy as the tensor contraction engine.
  warnings.warn(f'using {contract_engine} as the tensor contraction engine.')


In [2]:
# Molecular Information part
# !!!!! in CI part mo not forget the change electrons 
# !!!!! also orbitals are important  

xyz = '''
N   0.000   1.408   0.000
N   1.219  -0.704   0.000
N  -1.219  -0.704   0.000
B   0.000  -1.450   0.000
B  -1.256   0.725   0.000
B   1.256   0.725   0.000
H   0.000   2.414   0.000
H   2.090  -1.207   0.000
H  -2.090  -1.207   0.000
H   0.000  -2.642   0.000
H  -2.288   1.321   0.000
H   2.288   1.321   0.000'''
molecule = 'N3B3H6'
spin = 0
charge =  0


In [7]:
def rhf_calculation(xyz, molecule, spin, charge):
    mol = gto.M(
          atom= xyz,
          basis = 'sto-6g',
          charge = charge,
          verbose = 5 ,
          spin = spin, 
          symmetry = True,
          output  ="./"+molecule+".txt")

    mf = mol.RHF().run()
    mf.analyze() 
    S_ij = mf.get_ovlp()
    C_ij = mf.mo_coeff
    np.savetxt("C_"+molecule+'mo_coeff.txt', C_ij,fmt = '%.8f')
    np.savetxt('S_'+molecule+'_overlap.txt', S_ij,fmt = '%.8f')

    return mf, mol 

In [5]:
def orb_occ_table(orb_energy,orb_occ):
    Homo = None
    Lumo = None
    print ("orbital  energy   occ")

    for i in range (0,(len(orb_occ))):
        print(i ,"   ", orb_energy[i],"  ",orb_occ[i])
              
    for i in range(len(orb_occ) - 1):
        if orb_occ[i] == 2  and orb_occ[i+1] == 0:
            homo = i
            lumo = i + 1
            print(f"Homo_index: {homo}, Lumo_index: {lumo}")
    return homo, lumo


In [8]:
mf, mol  = rhf_calculation( xyz , molecule, spin, charge )
homo , lumo =orb_occ_table(mf.mo_energy, mf.mo_occ)

overwrite output file: ./N3B3H6.txt
orbital  energy   occ
0     -15.502352799321553    2.0
1     -15.502323176323495    2.0
2     -15.502260798975316    2.0
3     -7.510028484829517    2.0
4     -7.50998830295712    2.0
5     -7.509795040429394    2.0
6     -1.1341022826207723    2.0
7     -1.0647698602080804    2.0
8     -1.064657235935581    2.0
9     -0.6793087149433559    2.0
10     -0.6649720977278979    2.0
11     -0.6649184105147148    2.0
12     -0.5384289646452391    2.0
13     -0.5383580319199281    2.0
14     -0.532129629792065    2.0
15     -0.5113587151324148    2.0
16     -0.4563008981121525    2.0
17     -0.41038873579605784    2.0
18     -0.4103746542408875    2.0
19     -0.3375622045001431    2.0
20     -0.3375204476198551    2.0
21     0.3116588932923185    0.0
22     0.3117060337079195    0.0
23     0.46623030266013116    0.0
24     0.5742727364753637    0.0
25     0.6639758060386396    0.0
26     0.6640841734853786    0.0
27     0.6735918320680343    0.0
28     0.67

In [9]:
n_elec= 6
n_orb = 6 # number of orbital can be determined based on the number of pi  orbitals 
casscf = mcscf.CASSCF(mf, n_orb, n_elec)
'''output in order
e_tot
e_cas
Fci vecs 
mo_coeff
mo_energy'''
mo = casscf.sort_mo([17,20,21,22,23,24])# the numbers are from the index of pi orbitals 
casscf.kernel(mo) 


(np.float64(-240.52153395992195),
 np.float64(-6.771103324677995),
 FCIvector([[ 9.72446339e-01,  0.00000000e+00,  5.27340746e-02,
              0.00000000e+00, -5.27461264e-02,  0.00000000e+00,
             -8.71814495e-06,  5.32536838e-02,  0.00000000e+00,
              2.67668667e-05,  1.96312245e-05,  0.00000000e+00,
              3.12574609e-02,  2.40962846e-05,  0.00000000e+00,
              1.10758160e-02,  0.00000000e+00, -1.10774710e-02,
              0.00000000e+00,  5.72836373e-03],
            [ 0.00000000e+00, -1.20420919e-02,  0.00000000e+00,
             -3.09287808e-03,  0.00000000e+00,  5.26874641e-02,
              0.00000000e+00,  0.00000000e+00,  3.92413656e-03,
              0.00000000e+00,  0.00000000e+00,  1.51869446e-02,
              0.00000000e+00,  0.00000000e+00, -1.82430613e-03,
              0.00000000e+00,  3.97585547e-03,  0.00000000e+00,
             -2.96070737e-04,  0.00000000e+00],
            [ 5.27340746e-02,  0.00000000e+00, -6.41720277e-02,
     

In [10]:
casscf.fcisolver.max_cycle = 3000
casscf.canonicalize(mo_coeff=None, ci=None, eris=None, sort=False, cas_natorb=False, casdm1=None, verbose=3, with_meta_lowdin=False);

filename = f"{molecule}_Ci_coeffs.txt"

with open(filename, 'w', encoding='utf-8') as f:
    
    f.write("det-alpha,     det-beta,    CI coefficients\n")
    
    
    for c, ia, ib in casscf.fcisolver.large_ci(casscf.ci, n_orb, n_elec, tol=.000000001, return_strs=False):
        # Satır formatını oluşturalım
        line = f" {str(ia):<12} {str(ib):<12} {c:.15f}\n"
        
        #write in file 
        f.write(line)
        #show in terminal 
        print(line.strip())

[0 1 2]      [0 1 2]      0.972446338885777
[0 1 2]      [0 2 3]      0.052734074647319
[0 1 2]      [0 1 4]      -0.052746126437160
[0 1 2]      [1 2 4]      -0.000008718144953
[0 1 2]      [0 3 4]      0.053253683767864
[0 1 2]      [2 3 4]      0.000026766866686
[0 1 2]      [0 1 5]      0.000019631224502
[0 1 2]      [1 2 5]      0.031257460892080
[0 1 2]      [0 3 5]      0.000024096284580
[0 1 2]      [2 3 5]      0.011075816032241
[0 1 2]      [1 4 5]      -0.011077471036890
[0 1 2]      [3 4 5]      0.005728363734442
[0 1 3]      [0 1 3]      -0.012042091879582
[0 1 3]      [1 2 3]      -0.003092878077425
[0 1 3]      [0 2 4]      0.052687464064579
[0 1 3]      [1 3 4]      0.003924136560136
[0 1 3]      [0 2 5]      0.015186944640011
[0 1 3]      [1 3 5]      -0.001824306128046
[0 1 3]      [0 4 5]      0.003975855471303
[0 1 3]      [2 4 5]      -0.000296070737491
[0 2 3]      [0 1 2]      0.052734074647319
[0 2 3]      [0 2 3]      -0.064172027735972
[0 2 3]      [0 1 4]    